In [ ]:
# ========================================================
# 04_isolation_forest.ipynb
# Isolation Forest po rozszerzeniu normalnego datasetu
# ========================================================

import pandas as pd
import numpy as np
import joblib
import os
from sklearn.ensemble import IsolationForest

# ========================================================
# Ustawienia
# ========================================================

RANDOM_STATE = 42
CONTAMINATION = 0.05
N_ESTIMATORS = 500

print("=== ISOLATION FOREST BASELINE v2 - PO ROZSZERZENIU NORMAL DATASET ===")

# ========================================================
# 1. Ładowanie danych normalnych
# ========================================================

normal_df = pd.read_csv("../data/processed/normal_features.csv")
feature_columns = normal_df.columns.tolist()

print("Liczba cech:", len(feature_columns))
print("Liczba flowów normalnych:", len(normal_df))

X_normal = normal_df.values

# ========================================================
# 2. Trening Isolation Forest
# ========================================================

model_if = IsolationForest(
    n_estimators=N_ESTIMATORS,
    max_samples="auto",
    contamination=CONTAMINATION,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

model_if.fit(X_normal)

# score_samples: im mniejszy wynik, tym bardziej anomalna próbka
normal_scores = model_if.score_samples(X_normal)

threshold = np.percentile(normal_scores, CONTAMINATION * 100)

print("Próg detekcji Isolation Forest:", round(float(threshold), 6))

# ========================================================
# 3. Funkcja ewaluacji
# ========================================================

def evaluate_if(name):
    df = pd.read_csv(f"../data/processed/{name}_features.csv")
    df = df.reindex(columns=feature_columns, fill_value=0)

    X = df.values

    scores = model_if.score_samples(X)
    anomaly_score = -scores

    detected = int((scores < threshold).sum())
    total = len(scores)

    return {
        "scenario": name,
        "IF_mean_score": float(np.mean(anomaly_score)),
        "IF_median_score": float(np.median(anomaly_score)),
        "IF_max_score": float(np.max(anomaly_score)),
        "IF_detected": detected,
        "IF_total": total,
        "Isolation Forest DR (%)": round(detected / total * 100, 2)
    }

# ========================================================
# 4. Ewaluacja
# ========================================================

scenarios = [
    "guloader",
    "scanning",
    "njrat",
    "kongtuke1",
    "kongtuke2",
    "remcos",
    "xloader",
    "xworm",
    "phantomstealer"
]

results = []

# normal jako FPR
normal_anomaly_score = -normal_scores
normal_detected = int((normal_scores < threshold).sum())

results.append({
    "scenario": "normal (FPR)",
    "IF_mean_score": float(np.mean(normal_anomaly_score)),
    "IF_median_score": float(np.median(normal_anomaly_score)),
    "IF_max_score": float(np.max(normal_anomaly_score)),
    "IF_detected": normal_detected,
    "IF_total": len(normal_scores),
    "Isolation Forest DR (%)": round(normal_detected / len(normal_scores) * 100, 2)
})

print(
    f"{'normal (FPR)':15s} | detected: "
    f"{normal_detected:8d}/{len(normal_scores):<8d} "
    f"| DR: {round(normal_detected / len(normal_scores) * 100, 2):6.2f}%"
)

for sc in scenarios:
    res = evaluate_if(sc)
    results.append(res)

    print(
        f"{sc:15s} | detected: "
        f"{res['IF_detected']:8d}/{res['IF_total']:<8d} "
        f"| DR: {res['Isolation Forest DR (%)']:6.2f}% "
        f"| mean score: {res['IF_mean_score']:.6f}"
    )

df_if = pd.DataFrame(results)

# ========================================================
# 5. Zapis
# ========================================================

os.makedirs("../results", exist_ok=True)
df_if.to_csv("../results/isolation_forest_results.csv", index=False)

joblib.dump(model_if, "../models/isolation_forest_final.pkl")

print("\n=== WYNIKI ISOLATION FOREST ===")
print(df_if[[
    "scenario",
    "IF_detected",
    "IF_total",
    "Isolation Forest DR (%)"
]])

print("\nZapisano wyniki do: ../results/isolation_forest_results.csv")
print("Zapisano model do: ../models/isolation_forest_final.pkl")